In [1]:
# Load env variables

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [7]:
# Creating helper functions

# Helper functions to add user and assistant messages

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

In [4]:
def chat(messages, system=None, temperature=0.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
# Requesting Claude to generate structured data like JSON, Python code, or bulleted lists
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

# Get Claude's response
answer = chat(messages, temperature=0.0)

print(answer)

Here's a simple EventBridge rule in JSON:

```json
{
  "Name": "my-rule",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["stopped"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Id": "my-target",
      "Arn": "arn:aws:sns:us-east-1:123456789012:my-topic"
    }
  ]
}
```

This rule:
- **Triggers** when an EC2 instance is **stopped**
- **Sends** a notification to an **SNS topic**


In [ ]:
# Assistant Message Prefilling + Stop Sequences (*** Not supported in the latest Claude versions )
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

In [ ]:
# Option 1: Use output_config.format - when you need guaranteed structured output for production use

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule as JSON"}],
    output_config={
        "format": {
            "type": "json_schema",
            "json_schema": {
                "name": "eventbridge_rule",
                "schema": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "array", "items": {"type": "string"}},
                        "detail-type": {"type": "array", "items": {"type": "string"}},
                        "detail": {"type": "object"}
                    }
                }
            }
        }
    }
)

In [6]:
# Option 2: System prompt instruction - for quick scripts or when you don't know the schema in advance

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are a JSON-only API. Respond ONLY with valid JSON. No explanations, no markdown fences, no extra text.",
    messages=[{"role": "user", "content": "Generate a very short EventBridge rule as JSON"}]
)

In [ ]:
# Option 3: Move the prefill hint to the user turn

messages = [
    {"role": "user", "content": "Generate a very short EventBridge rule as JSON. Reply with only the raw JSON object, no markdown, no explanation."},
]

In [9]:
messages = []

add_user_message(messages, "Generate a very short EventBridge rule as JSON. Reply with only the raw JSON object, no markdown, no explanation.")

# Get Claude's response
answer = chat(messages, temperature=0.0)

print(answer)

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped"]
  }
}
